# Stage 4：Olist 訂單是否延遲預測

這本 Notebook 帶你完成「訂單核准時，預測最後是否超過預估送達日期」的流程。特徵與訓練邏輯由 `scripts/delay_features.py`、`scripts/train_delay_model.py` 共用，這裡負責導覽、檢查結果與示範推論。

請使用 **Shop (Python 3.12)** kernel。資料來自本機 `data/raw/olist/` 的六份 CSV，執行時不需要 MySQL。模型與大量逐筆預測保留在本機，Notebook 不內嵌這些檔案。


## 1. 先定義預測時點與答案

- **預測時點**：`order_approved_at`，也就是訂單／付款核准時間。
- **延遲標記**：實際送達的「日期」晚於預估送達的「日期」時，`delayed = 1`，其餘為 0。
- **答案可得時間**：`label_available_time` 是實際送達時間。訓練時必須檢查當時是否已知道答案。

例如預估 1 月 10 日送達，實際 1 月 10 日 23:59 送達仍算準時；1 月 11 日 00:00 送達才算延遲。Stage 3 用完整時間戳記比較午夜的統計方式與這裡不同，因此延遲率可能不同。

歷史訓練資料限於已送達、所需時間完整且時序合理的訂單。這個模型預測的是這群訂單的配送延遲，不包含取消或始終未送達的風險。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import joblib
import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "scripts" / "delay_features.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("請從專案根目錄或 notebooks 資料夾執行這本 Notebook。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.delay_features import (
    CATEGORICAL_FEATURES, FEATURES, NUMERIC_FEATURES, build_delay_dataset,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
MODEL_PATH = PROJECT_ROOT / "models" / "olist_delay_model.joblib"
METRICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "metrics.json"

print("專案根目錄：", PROJECT_ROOT)
print("Python：", sys.executable)


## 2. 建立每訂單一列的資料

商品明細與付款紀錄會各自彙總，再合併到訂單，避免一張訂單有多個商品與多筆付款時把金額重複加總。下面只顯示資料量、延遲比例與特徵摘要。


In [ ]:
dataset = build_delay_dataset(RAW_DIR)

display(pd.DataFrame([{
    "合格訂單數": dataset.shape[0],
    "資料欄位數": dataset.shape[1],
    "模型特徵數": len(FEATURES),
    "延遲訂單數": int(dataset["delayed"].sum()),
    "延遲比例": float(dataset["delayed"].mean()),
    "最早核准時間": dataset["prediction_time"].min(),
    "最晚核准時間": dataset["prediction_time"].max(),
}]))


## 3. 核准時可以知道哪些特徵

數值特徵包含商品數、商品與運費總額、付款資訊、商品重量與體積、預估配送天數、核准等待時間、購買時段及跨州商品比例。類別特徵包含顧客州別、主要賣家州別、主要品類與付款方式。完整欄位使用明確白名單 `FEATURES`。

實際送達、交給物流的時間、評論、最終訂單狀態，以及訂單／顧客 ID 都不進入模型。前幾項屬於預測之後才知道的資訊，ID 也不作為預測特徵。缺值補齊、標準化與類別編碼都放在 Pipeline，只用訓練集學習。

Olist 是事後快照資料，沒有每次商品、付款或預估交期變更的完整時間紀錄。本階段假設這些欄位在核准時已知；未來接真實系統時，應保存核准當下的快照來驗證這個假設。


In [ ]:
feature_summary = pd.DataFrame({
    "feature": FEATURES,
    "kind": ["numeric" if name in NUMERIC_FEATURES else "categorical"
             for name in FEATURES],
    "missing_rate": [float(dataset[name].isna().mean()) for name in FEATURES],
    "distinct_values": [dataset[name].nunique(dropna=True) for name in FEATURES],
})
display(feature_summary)


## 4. 依時間切分，排除當時還不知道的答案

訓練程式依 `prediction_time` 排序，以約 60%／20%／20% 的訂單量建立 train、validation、test；同一天的訂單全部放在同一區間，邊界日歸到後面的區間。

只做時間切分仍可能用到未來答案，因此還會做 **label maturity purge**：

- train 的送達時間必須嚴格早於 validation 開始時間。
- validation 的送達時間必須嚴格早於 test 開始時間。

因此三組實際筆數會受到日期邊界與 purge 影響。這次保留選定的原始 train 模型，不在選模後合併 validation 重訓；模型、前處理與門檻先固定，再對 test 預測一次。


## 5. 使用現有訓練成果，必要時由 CLI 建立

`RUN_TRAINING = False` 時，模型與報告都存在就直接讀取；任一檔案不存在時會自動執行訓練。若要重新產生整套成果，把設定改為 `True`。

CLI 使用目前 kernel 的 `sys.executable`，確保與 Notebook 使用同一個 Python 環境。


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING or not MODEL_PATH.is_file() or not METRICS_PATH.is_file():
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "train_delay_model.py"),
        "--data-dir", str(RAW_DIR),
        "--output-dir", str(METRICS_PATH.parent),
        "--model-path", str(MODEL_PATH),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("使用已存在的模型與 metrics.json。")


In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
artifact = joblib.load(MODEL_PATH)

if artifact["feature_names"] != list(FEATURES):
    raise ValueError("模型特徵與目前程式不同，請重新訓練。")
if artifact["selected_model"] != metrics["selected_model"]:
    raise ValueError("模型與報告不一致，請重新產生同一批訓練成果。")

split_info = metrics["metadata"]["split"]
print("Validation 開始：", split_info["validation_start"])
print("Test 開始：", split_info["test_start"])
display(pd.DataFrame.from_dict(split_info["partitions"], orient="index"))


## 6. 在 validation 選模型與決策門檻

這次比較 Dummy baseline、Logistic Regression、Random Forest。Dummy 只作為基本參考；可選模型是 Logistic Regression 與 Random Forest，分別學習線性關係與樹模型的非線性關係。

先在兩個可選模型中，以 validation 的 **Average Precision（AP）最高者選模型**，並檢查 AP 是否高於 Dummy。AP 衡量排序能力，在延遲訂單較少的資料中特別有參考價值。再只使用被選模型的 validation 分數，選擇 **F1 最高的門檻**。F1 同時考慮 precision 與 recall；這是本階段的實驗策略，未來可依人工查核量或延遲漏判成本調整。

下表比較候選模型的 validation 結果。這些分類指標使用 0.5 門檻；AP 和 ROC AUC 使用連續分數，不受分類門檻影響。


In [ ]:
validation_rows = []
for model_name, result in metrics["model_comparison"].items():
    row = {"model": model_name, "status": result["status"]}
    if result["status"] == "ok":
        row.update(result["validation_at_0_5"])
        row["fit_seconds"] = result["fit_seconds"]
    else:
        row["error"] = result.get("error", "")
    validation_rows.append(row)

models_validation = pd.DataFrame(validation_rows)
display(models_validation)


In [ ]:
selected = pd.DataFrame([{
    "selected_model": artifact["selected_model"],
    "threshold": float(artifact["threshold"]),
    "selection_metric": "validation average_precision",
    "threshold_objective": "validation F1",
    "validation_AP": metrics["validation"]["average_precision"],
    "validation_AP_above_dummy": metrics["metadata"]["model_selection"]["validation_ap_above_dummy"],
    "validation_F1_at_selected_threshold": metrics["validation"]["f1"],
}])
display(selected)


## 7. 最後才看 test，記錄保留資料的表現

Test 不參與補值、編碼、模型選擇或門檻選擇。下面讀取凍結 Pipeline 在 test 的一次評估結果，並與全猜準時的固定 baseline 比較，不使用 test 結果反覆調參。

- **Precision**：判成延遲的訂單中，實際延遲的比例。
- **Recall**：實際延遲的訂單中，被模型抓到的比例。
- **F1**：precision 與 recall 的調和平均。
- **AP／ROC AUC**：衡量連續分數的排序能力。
- **Accuracy／balanced accuracy**：作為補充，需一起考慮延遲與準時的比例。


In [ ]:
test_keys = [
    "rows", "positive_rate", "threshold", "average_precision", "roc_auc",
    "f1", "recall", "precision", "balanced_accuracy", "accuracy",
]
testmetrics = pd.DataFrame([
    {"model": artifact["selected_model"],
     **{key: metrics["test"][key] for key in test_keys}},
    {"model": "all_on_time_baseline",
     **{key: metrics["test_baseline"][key] for key in test_keys}},
])
display(testmetrics)

confusion = pd.DataFrame(
    metrics["test"]["confusion_matrix"],
    index=["實際準時", "實際延遲"],
    columns=["預測準時", "預測延遲"],
)
display(confusion)
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_test_confusion_matrix.png"
)))


## 8. 看哪些特徵影響 validation 的排序

Permutation importance 會打亂某一欄，再看 validation AP 下降多少。下降越多，代表被選模型較依賴該欄；負值或接近零代表這次抽樣沒有顯示穩定的正面貢獻。

圖表使用 validation 抽樣，未用 test 來挑特徵。這是模型行為的描述，不能當成延遲的因果證據；高度相關的欄位也可能分攤重要性。


In [ ]:
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_validation_permutation_importance.png"
)))


## 9. 用一筆資料示範 Pipeline 推論

這裡取歷史資料的一筆來示範介面，只把白名單特徵交給儲存的 Pipeline。前處理會沿用訓練時學到的設定；決策門檻沿用 artifact 內 validation 選出的值。

`predict_proba` 輸出作為延遲分數使用。這次沒有做機率校準，分數不應直接解讀成已校準的真實延遲機率。這一筆示範也不構成額外的效能評估。


In [ ]:
example = dataset.iloc[[0]]
feature_row = example.loc[:, artifact["feature_names"]].copy()
pipeline = artifact["pipeline"]
positive_class_index = list(pipeline.classes_).index(1)
delay_score = float(pipeline.predict_proba(feature_row)[0, positive_class_index])
threshold = float(artifact["threshold"])

display(pd.DataFrame([{
    "order_id": example["order_id"].iloc[0],
    "model": artifact["selected_model"],
    "delay_score_uncalibrated": delay_score,
    "threshold": threshold,
    "predicted_delayed": int(delay_score >= threshold),
}]))


## 10. 可選：比較更多模型的開發區間時間回測

新的比較加入 Extra Trees、HistGradient Boosting、XGBoost、LightGBM，連同 Dummy、Logistic Regression 與 Random Forest，一共比較七種模型。報告放在 `reports/stage4/model_comparison.json`，這個區塊只讀取既有成果，缺檔時會顯示執行指令。

這份報告使用三個**開發區間回測**，各自建立 train、validation、evaluation。Validation 與 evaluation 之間留一個月等待區間，讓 validation 的送達標記成熟；等待區間新核准的訂單不加入 validation 或 evaluation。Train 的標記必須在 validation 開始前已知，validation 的標記則必須在 evaluation 開始前已知。各模型在 validation 選門檻後，才評估該 fold 的 evaluation；Dummy 保持固定 0.5 門檻作為參考。

回測的 evaluation 只包含原始固定 test 開始時間（2018-05-26）之前核准的訂單，但保留這些訂單事後才取得的送達答案。因此它是事後整理的歷史評估，並非 evaluation 月底當時就能知道全部結果。

**原先 Random Forest 的固定 test 結果與模型 artifact 保持原樣。** 回測平均 AP 比較反映開發區間的表現，與前面固定 test 的單次評估不同，不能把兩者的指標直接當成同一批訂單的結果。


In [ ]:
COMPARISON_PATH = PROJECT_ROOT / "reports" / "stage4" / "model_comparison.json"
COMPARISON_IMAGE = (
    PROJECT_ROOT / "reports" / "images" / "stage4_backtest_model_comparison.png"
)

if COMPARISON_PATH.is_file():
    comparison_report = json.loads(COMPARISON_PATH.read_text(encoding="utf-8"))
    print("開發區間時間回測報告：", comparison_report["status"])
    comparison_table = pd.json_normalize(comparison_report["summary"])
    comparison_columns = {
        "display_name": "model",
        "status": "status",
        "successful_folds": "successful_folds",
        "development_ap_rank": "development_AP_rank",
        "metrics.average_precision.mean": "AP_mean",
        "metrics.average_precision.std": "AP_std",
        "metrics.roc_auc.mean": "ROC_AUC_mean",
        "metrics.f1.mean": "F1_mean",
        "metrics.recall.mean": "recall_mean",
        "metrics.precision.mean": "precision_mean",
        "metrics.ap_lift_over_prevalence.mean": "AP_lift_over_prevalence_mean",
    }
    display(comparison_table.reindex(columns=comparison_columns).rename(
        columns=comparison_columns
    ))
    if COMPARISON_IMAGE.is_file():
        display(Image(filename=str(COMPARISON_IMAGE)))
else:
    compare_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "compare_delay_models.py"),
    ]
    print("尚未產生額外模型比較；如要執行，請在終端機使用：")
    print(subprocess.list2cmdline(compare_command))


## 11. 減少誤報：平衡模式與歷史特徵實驗

使用者選定的目標是：validation 至少抓到 30% 延遲訂單，再盡量減少提醒量。新版比較 RF／Logistic、原 21 欄／加入 10 個過去 90 天已送達特徵，所有選模與門檻選擇都只用 validation。

歷史窗口為 `[核准時間−90天, 核准時間)`，答案必須當時已知，同 timestamp 或未來結果不能使用。結果會隨已送達事件更新，但模型不在 evaluation 重訓。

本次歷史特徵沒有勝出，最終平衡 artifact 仍為 static RF。逐月自動選模雖降低合計誤報，卻有一個月召回僅 1.12%，不能只看合計。因此保留原版預設；要使用新門檻，執行 `predict_delay.py --profile balanced`。

下方合計 precision／recall 由加總 TP／FP／FN 計算，與前面每月指標的平均不同。最終保留期間已檢視過，新結果僅是回溯診斷，不是新的獨立測試。缺檔時只顯示指令，不自動重新訓練。

In [ ]:
ALERT_PATH = PROJECT_ROOT / "reports" / "stage4" / "alert_optimization.json"
if ALERT_PATH.is_file():
    alert_report = json.loads(ALERT_PATH.read_text(encoding="utf-8"))
    print("Validation 最低召回目標：", alert_report["min_validation_recall"])
    display(pd.DataFrame(alert_report["summary"])[[
        "model", "policy", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "false_positive_reduction_vs_rf_f1",
    ]])
    display(pd.DataFrame([{
        "month": fold["name"],
        "selected_model": fold["experiment"]["selected_model"],
        "evaluation_recall": fold["experiment"]["models"][fold["experiment"]["selected_model"]]["evaluation"]["balanced"]["recall"],
    } for fold in alert_report["folds"]]))
    final_experiment = alert_report["final"]["experiment"]
    final_name = final_experiment["selected_model"]
    final_model = final_experiment["models"][final_name]
    print("最終候選：", final_name, "門檻：", final_model["thresholds"]["balanced"])
    print("以下為已檢視保留資料的回溯診斷，非新的獨立測試。")
    display(pd.DataFrame([{
        "profile": label, "FP": metrics["confusion_matrix"][0][1],
        "TP": metrics["confusion_matrix"][1][1], "FN": metrics["confusion_matrix"][1][0],
        "precision": metrics["precision"], "recall": metrics["recall"],
    } for label, metrics in (
        ("original", final_experiment["models"]["random_forest_static"]["evaluation"]["f1"]),
        ("balanced", final_model["evaluation"]["balanced"]),
    )]))
    alert_image = PROJECT_ROOT / "reports" / "images" / "stage4_alert_tradeoff.png"
    if alert_image.is_file():
        display(Image(filename=str(alert_image)))
else:
    print("請先執行：", subprocess.list2cmdline([
        sys.executable, str(PROJECT_ROOT / "scripts" / "improve_delay_alerts.py"),
        "--min-recall", "0.30", "--n-jobs", "4",
    ]))


## 12. 可選：月份落差與誤報診斷

這裡讀取 `reports/stage4/monthly_error_diagnostics.json`，回溯檢查已看過的 holdout。診斷使用儲存的平衡模型與**固定決策門檻**，各月都沿用同一門檻；沒有重新訓練、依月份調門檻，或改善模型。這些結果屬於既有保留資料的事後分析，不是新的獨立測試。

月表列出誤報 FP、抓到的延遲 TP、漏掉的延遲 FN 與召回率。**FPR 的分母是實際準時訂單數**，不是提醒數；`partial_month = True` 表示只涵蓋部分月份，不能直接拿件數和完整月比較。

路線表先保留 `sufficient_on_time_rows` 為真的群組，再依 FP 件數取前五名，同時列出準時訂單數與提醒數。即使通過最低樣本量篩選，也要留意分母與小樣本波動。承諾天數使用核准到預估送達的曆日差，實際天數使用核准到送達的連續天數，兩者是描述統計，不能直接替代曆日延遲標記。

這些分組與月份差異描述的是關聯，不能據此判定州別、路線或交期造成誤報。報告缺少時，下方只會顯示產生診斷的指令。


In [ ]:
DIAGNOSTICS_PATH = (
    PROJECT_ROOT / "reports" / "stage4" / "monthly_error_diagnostics.json"
)
if DIAGNOSTICS_PATH.is_file():
    diagnostic_report = json.loads(DIAGNOSTICS_PATH.read_text(encoding="utf-8"))
    print("回溯診斷模型：", diagnostic_report["model"]["selected_model"])
    print("固定門檻：", diagnostic_report["model"]["threshold"])

    month_columns = [
        "month", "rows", "on_time_rows", "false_positives", "false_positive_rate",
        "true_positives", "false_negatives", "recall", "partial_month",
    ]
    monthly_errors = pd.DataFrame(diagnostic_report["test_months"])
    display(monthly_errors[month_columns].rename(columns={
        "false_positives": "FP", "false_positive_rate": "FPR",
        "true_positives": "TP", "false_negatives": "FN",
    }))

    print("準時樣本數足夠的路線：依 FP 件數列前五名")
    route_table = pd.DataFrame(diagnostic_report["groups"]["state_route"])
    if route_table.empty:
        print("沒有可列出的路線群組。")
    else:
        route_top5 = route_table.loc[
            route_table["sufficient_on_time_rows"].eq(True)
        ].sort_values(
            ["false_positives", "false_positive_rate"], ascending=False
        ).head(5)
        display(route_top5[[
            "state_route", "rows", "on_time_rows", "alert_rows",
            "false_positives", "false_positive_rate", "precision", "recall",
        ]].rename(columns={"false_positives": "FP", "false_positive_rate": "FPR"}))

    print("各月承諾與實際送達天數：")
    delivery_days_summary = pd.DataFrame([{
        "month": row["month"],
        "rows": row["rows"],
        "promise_days_mean": row["feature_and_outcome_summary"]["promise_days_mean"],
        "actual_delivery_days_mean": row["feature_and_outcome_summary"]["actual_delivery_days_mean"],
        "partial_month": row["partial_month"],
    } for row in diagnostic_report["test_months"]])
    display(delivery_days_summary)

    for image_name in (
        "stage4_monthly_errors.png", "stage4_false_positive_groups.png"
    ):
        diagnostic_image = PROJECT_ROOT / "reports" / "images" / image_name
        if diagnostic_image.is_file():
            display(Image(filename=str(diagnostic_image)))
else:
    print("尚未產生月份診斷；請在終端機執行：")
    print(subprocess.list2cmdline([
        str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"),
        str(PROJECT_ROOT / "scripts" / "diagnose_delay_errors.py"),
    ]))


## 13. 可選：等待期與路線承諾特徵實驗

這裡只讀取 `reports/stage4/waiting_strategy_comparison.json` 與既有圖表，不會啟動訓練。**60 天是看結果前指定的主要等待期，30 天是敏感度分析**，不是看 evaluation 成績後挑較好的等待期。每個等待期比較 Random Forest／Logistic Regression、原始特徵／路線承諾特徵，另列 validation 自動選模策略，共 10 種策略。

以 evaluation 開始日 E 為例，validation 核准區間為 `[E−等待天數−30天, E−等待天數)`；train 的核准截止日再比 validation 開始早一個等待期。Train 標籤必須嚴格早於 validation 開始，validation 標籤必須嚴格早於 E。**等待期改變時，train 與 validation 日期、訓練量及資料新近程度也一起改變**，因此比較結果不能解釋成單純延長等待期的因果效果。

路線特徵以主賣家州別 → 顧客州別區分方向；90 天歷史只用核准當時已知的有效送達事件。送達天數分位數與承諾天數都採**曆日尺度**。歷史會隨時間經過及送達結果取得而動態更新，但各折模型不在 evaluation 重新訓練。有效已送達 cohort 仍不包含所有未送達、取消或其他無效訂單，不能把成熟標籤保留率當成所有核准訂單的結果涵蓋率。

所有模型與門檻只用 train／validation 決定；**30% recall 是 validation 的最低目標，不保證後續月份達標**。下方逐月列出 validation 延遲正例保留率與被排除正例數，合計 precision／recall 則由加總 TP／FP／FN 計算，並另列最差月份 recall，避免只看合計。

這三個開發月份已在先前實驗檢視過，屬於開發回測，不是新的未知 test。**本次實驗不評估 2018-05-26 起的原 holdout**。研究 artifact 另存於 `models/research/`，未替換原版或平衡模式的推論預設。


In [ ]:
WAITING_PATH = (
    PROJECT_ROOT / "reports" / "stage4" / "waiting_strategy_comparison.json"
)
if WAITING_PATH.is_file():
    waiting_report = json.loads(WAITING_PATH.read_text(encoding="utf-8"))
    print("預先指定的主要等待期（天）：", waiting_report["primary_wait_days_prespecified"])
    print("Validation 最低召回目標：", waiting_report["min_validation_recall"])
    print("以下為已檢視開發月份的回測；本次不評估原 holdout，也不替換推論預設。")

    waiting_summary = pd.DataFrame(waiting_report["summary"])
    display(waiting_summary[[
        "wait_days", "model", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_month_recall",
    ]].rename(columns={
        "false_positives": "FP", "true_positives": "TP", "false_negatives": "FN",
    }))

    waiting_fold_rows = []
    for fold in waiting_report["folds"]:
        split = fold["split"]
        validation = split["partitions"]["validation"]
        waiting_fold_rows.append({
            "wait_days": fold["wait_days"],
            "month": fold["name"],
            "selected_model": fold["experiment"]["selected_model"],
            "train_prediction_end": split["train_prediction_end"],
            "validation_start": split["validation_start"],
            "validation_end": split["validation_end"],
            "validation_positive_rows_before_purge": validation["positive_rows_before_purge"],
            "validation_positive_retention_rate": validation["positive_retention_rate"],
            "validation_purged_positive_rows": validation["purged_positive_rows"],
        })
    print("有效已送達 cohort 內的標籤成熟情況；非所有核准訂單涵蓋率：")
    display(pd.DataFrame(waiting_fold_rows))

    waiting_image = PROJECT_ROOT / "reports" / "images" / "stage4_waiting_tradeoff.png"
    if waiting_image.is_file():
        display(Image(filename=str(waiting_image)))
else:
    print("python scripts/compare_waiting_strategies.py --n-jobs 4")


## 14. 可選：多月份驗證

本節只讀取 `reports/stage4/multimonth_validation.json`，不啟動訓練。固定至少 60 天等待，取連續三個完整日曆月作 validation；同一模型在最早驗證月開始時訓練，之後固定。訓練核准截止再早 60 天，訓練標籤嚴格早於模型訓練日，驗證標籤嚴格早於 evaluation 開始日。

預先指定的主策略 `every_month_floor` 使用同一門檻，要求每個驗證月份 recall ≥30%，再選提醒量最少的候選。共同門檻是各月最高可行門檻的最小值。對照 `pooled_3months` 只要求三月合計達標；兩者使用完全相同 train、模型、分數及全三月選模母體。`recent_month` 只用最新驗證月決定門檻與選模，作描述性參考。

擴大 validation 會讓訓練資料更舊、筆數更少，舊單月實驗只作背景。各 fold 可能重用歷史月份，早期 evaluation 在之後 fold 成為 validation；這是滾動開發回測，不是獨立未知測試。本輪仍不評估 2018-05-26 起的原 holdout。**每個驗證月達標也不保證後續月份達標**；研究模型另存，不自動替換預設。


In [ ]:
MULTIMONTH_PATH = PROJECT_ROOT / "reports" / "stage4" / "multimonth_validation.json"
if MULTIMONTH_PATH.is_file():
    multimonth_report = json.loads(MULTIMONTH_PATH.read_text(encoding="utf-8"))
    print("主策略：", multimonth_report["primary_strategy_prespecified"])
    print("已檢視開發月份回測；每月驗證達標不保證後續達標。")
    display(pd.DataFrame(multimonth_report["summary"])[[
        "strategy", "model", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_evaluation_month_recall",
        "minimum_validation_month_recall",
    ]])
    multimonth_selections, multimonth_coverage = [], []
    for fold in multimonth_report["folds"]:
        for strategy, candidate in fold["experiment"]["selected_models"].items():
            decision = fold["experiment"]["models"][candidate]["strategies"][strategy]
            multimonth_selections.append({
                "evaluation_month": fold["name"], "strategy": strategy, "model": candidate,
                "threshold": decision["threshold"],
                "validation_month_recalls": {
                    month: metrics["recall"] for month, metrics in decision["validation_by_month"].items()
                },
                "evaluation_recall": decision["evaluation"]["recall"],
            })
        for month in fold["split"]["validation_months"]:
            multimonth_coverage.append({
                "evaluation_month": fold["name"], "validation_month": month["month"],
                "positive_rows_before_purge": month["positive_rows_before_purge"],
                "positive_rows": month["positive_rows"],
                "positive_retention_rate": month["positive_retention_rate"],
            })
    display(pd.DataFrame(multimonth_selections))
    print("標籤保留率分母限有效已送達 cohort：")
    display(pd.DataFrame(multimonth_coverage))
    multimonth_image = PROJECT_ROOT / "reports" / "images" / "stage4_multimonth_validation.png"
    if multimonth_image.is_file():
        display(Image(filename=str(multimonth_image)))
else:
    print("python scripts/compare_multimonth_validation.py --n-jobs 4")


## 15. 可選：每月重訓的滾動驗證

本節只讀取 `reports/stage4/monthly_retraining.json`，不啟動訓練。每個月 M 第一日以核准時間 `<M−60天`、結果時間 `<M` 的訂單重訓，再為 M 月訂單留下當時的 out-of-time 分數。共用的歷史月份只訓練一次；同一月份在不同 evaluation E 使用時，仍須重新按 `label_available_time < E` 篩成熟結果。

E 時依前三個成熟完整月的舊版模型分數選門檻與候選，再使用 E 當天重訓的模型預測。**最終模型可以訓練到歷史 validation 訂單，但不拿它重算這些訂單的驗證分數**。對照最新月、三月合計及預先指定的每月 recall ≥30% 主策略，原始分數跨版本共用門檻；這些分數未校準，不保證不同模型版本具有同一尺度。

逐月門檻、分數分布與後續月份的指標只用來診斷轉移效果，不依 evaluation 調整規則。本輪仍沿用已檢視的開發月份，不評估原 2018-05-26 起 holdout，也不自動提升研究模型為預設。


In [ ]:
ROLLING_PATH = PROJECT_ROOT / "reports" / "stage4" / "monthly_retraining.json"
if ROLLING_PATH.is_file():
    rolling_report = json.loads(ROLLING_PATH.read_text(encoding="utf-8"))
    print("主策略：", rolling_report["primary_strategy_prespecified"])
    print("Fit 次數：", rolling_report["fit_count"])
    display(pd.DataFrame(rolling_report["summary"])[[
        "strategy", "model", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_evaluation_month_recall", "minimum_validation_month_recall",
    ]])
    display(pd.DataFrame([{
        "model_month": row["month"], "fit_time": row["split"]["model_fit_time"],
        "train_approval_end": row["split"]["train_prediction_end"],
        "train_rows": row["split"]["partitions"]["train"]["rows"],
        "train_positive_retention": row["split"]["partitions"]["train"]["positive_retention_rate"],
    } for row in rolling_report["monthly_training"]]))
    rolling_selections = []
    for fold in rolling_report["folds"]:
        for strategy, candidate in fold["experiment"]["selected_models"].items():
            model = fold["experiment"]["models"][candidate]
            decision = model["strategies"][strategy]
            rolling_selections.append({
                "evaluation_month": fold["name"], "strategy": strategy, "model": candidate,
                "threshold": decision["threshold"],
                "past_month_thresholds": model["score_transfer_diagnostics"]["individual_validation_month_thresholds"],
                "minimum_validation_recall": decision["minimum_validation_month_recall"],
                "evaluation_recall": decision["evaluation"]["recall"],
                "historical_validation_rows_now_in_train": fold["historical_validation_rows_in_final_train"],
            })
    display(pd.DataFrame(rolling_selections))
    print("歷史驗證分數由舊版模型產生；未以最終模型重新計算。跨版本門檻不保證後續召回。")
    rolling_image = PROJECT_ROOT / "reports" / "images" / "stage4_monthly_retraining.png"
    if rolling_image.is_file():
        display(Image(filename=str(rolling_image)))
else:
    print("python scripts/compare_monthly_retraining.py --n-jobs 4")


## 16. 可選：分數校準與每日提醒上限

本節只讀取 `reports/stage4/calibration_budget.json`，不啟動訓練。沿用既有月版模型的 OOT 分數：每個 E 的前三個歷史月，前兩月 fit 校準器，第三月選門檻與候選，三個月都只使用標籤時間 `< E` 的結果。校準與門檻在 E 決定，並非在第三個歷史月起點就部署。

比較原分數、Sigmoid、Isotonic；主方案事先指定 Sigmoid／每日最多 10%，另看 5%、20% 與無上限。每日容量為 `floor(比例 × 當日完整評分 cohort)`，未成熟標籤仍參與容量與排名，但不拿未知結果算驗證指標。同分依校準分數、原分數、訂單 ID 排序。30% 召回不可行時，保留上限並明確標示，不宣稱達標。

每日批次結束才排名，可能等待一天；歷史 cohort 仍只含有效完成訂單。推論要有完整每日批次，使用 `--complete-daily-batches` 明確承諾完整性；不能從單張訂單保證每日上限。校準指標區分 fit 描述、獨立政策驗證及開發回測，單調校準本身不能改善排序。本輪沿用已檢視的開發月份，未評分原 holdout，也未替換預設模型。


In [ ]:
CALIBRATION_PATH = PROJECT_ROOT / "reports" / "stage4" / "calibration_budget.json"
if CALIBRATION_PATH.is_file():
    calibration_report = json.loads(CALIBRATION_PATH.read_text(encoding="utf-8"))
    print("主方案：", calibration_report["primary_method_prespecified"], calibration_report["primary_budget_prespecified"])
    print("新基礎模型 fit：", calibration_report["base_model_fit_count"], "校準器 fit：", calibration_report["learned_calibrator_fit_count"])
    summary_frame = pd.DataFrame(calibration_report["summary"])
    display(summary_frame.loc[summary_frame.model.eq("validation_selected"), [
        "method", "budget", "false_positives", "true_positives", "false_negatives",
        "alerts", "precision", "recall", "minimum_evaluation_month_recall", "infeasible_validation_months",
    ]])
    calibration_quality, primary_decisions = [], []
    for fold in calibration_report["folds"]:
        for method in calibration_report["methods"]:
            metrics = fold["experiment"]["models"]["random_forest_route"][method]["probability_metrics"]
            for phase, values in metrics.items():
                calibration_quality.append({
                    "evaluation_month": fold["name"], "candidate": "random_forest_route", "method": method,
                    "phase": phase, "rows": values["rows"], "brier": values["brier_score"],
                    "log_loss": values["log_loss"], "ece": values["expected_calibration_error"],
                })
        method, budget = calibration_report["primary_method_prespecified"], calibration_report["primary_budget_prespecified"]
        candidate = fold["experiment"]["selected_models"][method][budget]
        decision = fold["experiment"]["models"][candidate][method]["strategies"][budget]
        primary_decisions.append({
            "evaluation_month": fold["name"], "candidate": candidate, "threshold": decision["threshold"],
            "validation_feasible": decision["threshold_policy"]["feasible"],
            "maximum_validation_recall": decision["threshold_policy"]["max_achievable_recall"],
            "evaluation_recall": decision["evaluation"]["recall"],
            "evaluation_alerts": decision["evaluation_batch"]["alerts"],
            "maximum_daily_alert_rate": decision["evaluation_batch"]["maximum_daily_alert_rate"],
        })
    display(pd.DataFrame(primary_decisions))
    display(pd.DataFrame(calibration_quality))
    print("10% 上限的三組歷史驗證都無法達到30%召回；減少誤報同時增加漏報。開發回測不是未知測試。")
    calibration_image = PROJECT_ROOT / "reports" / "images" / "stage4_calibration_budget.png"
    if calibration_image.is_file():
        display(Image(filename=str(calibration_image)))
else:
    print("先產生每月重訓的模型／分數，再執行 python scripts/compare_calibration_budget.py")


## 17. 可選：固定每日 10% 預算的漏報診斷與排名改善

本節只讀取 `budget_miss_diagnostics.json` 與 `budget_ranking.json`，不啟動訓練、不新增圖表。基準主方案的漏報先按路線、承諾天數與每日排名分組。絕對漏報數也受群組訂單量影響，需同時看延遲總量、召回率與小樣本標記；這些是開發資料事後診斷。

六個新候選及參數在執行前固定，包含 Route／45 欄風險特徵的 LightGBM、XGBoost，加權 LightGBM 與風險特徵 RF。每個月 M 仍只用核准 `< M−60天` 且結果 `< M` 的訂單訓練；新增的賣家、路線歷史只看各訂單核准前已送達的事件。ZIP3 是類別，並非距離。

每日完整批次排名取 `floor(10% × 當日有效評分筆數)`，門檻固定 0。三個歷史 OOT 月的未知結果仍占名額；只有結果 `< E` 的列進驗證。依最差歷史月召回、合計 TP、AP 與固定候選順序選模，固定後才計算 E 當月結果。30% 歷史下限與開發穩定度分開呈現；本輪沿用已檢視的開發月，原 holdout 未評分，不保證未知資料成績。Dashboard 依使用者要求暫緩。


In [ ]:
MISS_PATH = PROJECT_ROOT / "reports" / "stage4" / "budget_miss_diagnostics.json"
RANKING_PATH = PROJECT_ROOT / "reports" / "stage4" / "budget_ranking.json"
if MISS_PATH.is_file():
    miss_report = json.loads(MISS_PATH.read_text(encoding="utf-8"))
    baseline_groups = miss_report["diagnostics"]["aggregate"]["groups"]
    print("基準開發漏報路線；群組訂單量也會影響絕對漏報數：")
    display(pd.DataFrame(baseline_groups["state_route"]).head(10))
    display(pd.DataFrame(baseline_groups["promise_days"]))
if RANKING_PATH.is_file():
    ranking_report = json.loads(RANKING_PATH.read_text(encoding="utf-8"))
    print("每日固定預算：", ranking_report["daily_cap"])
    print("新模型 fit：", ranking_report["new_fit_count"], "沿用歷史版本：", ranking_report["reused_legacy_version_count"])
    display(pd.DataFrame(ranking_report["summary"])[[
        "model_or_selection", "selected_candidates", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    ranking_progress = []
    for fold in ranking_report["folds"]:
        candidate = fold["experiment"]["selected_models"][ranking_report["primary_selection_prespecified"]]
        decision = fold["experiment"]["models"][candidate]
        ranking_progress.append({
            "evaluation_month": fold["name"], "candidate": candidate,
            "minimum_historical_recall": decision["minimum_validation_month_recall"],
            "historical_floor_met": decision["all_validation_months_meet_30_percent"],
            "development_recall": decision["development_evaluation"]["recall"],
            "development_alerts": decision["evaluation_batch"]["alerts"],
        })
    display(pd.DataFrame(ranking_progress))
    print("適用程度：", ranking_report["suitability"])
    new_groups = ranking_report["selected_development_miss_diagnostics"]["aggregate"]["groups"]
    display(pd.DataFrame(new_groups["state_route"]).head(10))
    display(pd.DataFrame(new_groups["promise_days"]))
    print("歷史驗證選模與開發事後診斷分開；尚無新的獨立測試成績。")
else:
    print("python scripts/diagnose_budget_misses.py")
    print("python scripts/improve_budget_ranking.py --n-jobs 4")


## 18. 可選：待件負載與每日排序模型

本節只讀取第二輪 `pending_ranking.json`，不訓練、不新增圖表。完整方法見 `docs/STAGE4_PENDING_RANKING.md`。

保留第一輪 45 欄，另加全域／賣家／路線的 30、90 天待件數、30 天平均等待及兩種逾期數，共 60 欄。母體包含當時已核准的有效原始訂單，不依最終狀態過濾；送達嚴格早於查詢才移除。缺少取消事件時間，取消單可能留在窗口內，這是資料限制。

新增 Logistic、LightGBM、XGBoost 及兩種 LambdaRank，共 40 次月份訓練；前輪 80 版本重載核對。LambdaRank 以成熟訓練列的核准日分組，二元增益與 truncation 20 的 NDCG 目標只是每日 10% 召回的代理；sigmoid(margin) 是單調分數介面，不是機率校準。

仍固定每日 10% 與三個歷史月最差召回優先選模，固定後才看開發月。整體設計已參考反覆檢視的開發診斷，因此不能稱獨立測試；原 holdout 不評分。適用門檻未通過時，保留研究模型並繼續改善，Dashboard 暫緩。


In [ ]:
PENDING_PATH = PROJECT_ROOT / "reports" / "stage4" / "pending_ranking.json"
if PENDING_PATH.is_file():
    pending_report = json.loads(PENDING_PATH.read_text(encoding="utf-8"))
    print("每日上限：", pending_report["daily_cap"], "新增 fit：", pending_report["new_fit_count"])
    display(pd.DataFrame(pending_report["summary"])[[
        "model_or_selection", "selected_candidates", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    pending_progress = []
    for fold in pending_report["folds"]:
        candidate = fold["experiment"]["selected_models"][pending_report["primary_selection_prespecified"]]
        decision = fold["experiment"]["models"][candidate]
        pending_progress.append({
            "month": fold["name"], "candidate": candidate,
            "minimum_historical_recall": decision["minimum_validation_month_recall"],
            "historical_floor_met": decision["all_validation_months_meet_30_percent"],
            "development_recall": decision["development_evaluation"]["recall"],
            "development_alerts": decision["evaluation_batch"]["alerts"],
        })
    display(pd.DataFrame(pending_progress))
    print("適用程度：", pending_report["suitability"])
    pending_groups = pending_report["selected_development_miss_diagnostics"]["aggregate"]["groups"]
    display(pd.DataFrame(pending_groups["state_route"]).head(10))
    display(pd.DataFrame(pending_groups["promise_days"]))
else:
    print("python scripts/improve_pending_ranking.py --n-jobs 4")


## 19. 可選：訓練答案取得時間與等待期

本節只讀 `outcome_clock.json`，不啟動訓練、不新增圖表。比較 Logistic／原特徵、RF／路線、LightGBM／待件，交叉三種訓練設計共 72 fit：實際送達時鐘＋0 天等待、承諾到期時鐘＋0 天等待、承諾到期時鐘＋30 天等待。

只在訓練成熟篩選新增 `training_outcome_known_time = min(實際送達, 承諾日期次日午夜)`。延遲答案在逾期時即可確認，準時答案在配送時確認；原本 `label_available_time`、配送歷史、特徵、驗證成熟篩選與每日分母全部保留。九個新候選固定參數，連同前輪 15 候選依相同歷史規則選模；開發月結果不決定參數或等待期。

仍限最終有效已送達群體，不能據此推論每張原始待件將會配送；也缺少事件入庫時間與承諾變更紀錄。所有月份及整體研究設計已參考既有開發診斷，不是新的獨立測試。歷史門檻與開發穩定度分開呈現。


In [ ]:
CLOCK_PATH = PROJECT_ROOT / "reports" / "stage4" / "outcome_clock.json"
if CLOCK_PATH.is_file():
    clock_report = json.loads(CLOCK_PATH.read_text(encoding="utf-8"))
    print("每日上限：", clock_report["daily_cap"], "新增 fit：", clock_report["new_fit_count"])
    display(pd.DataFrame(clock_report["summary"])[[
        "model_or_selection", "selected_candidates", "false_positives", "true_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    clock_progress = []
    for fold in clock_report["folds"]:
        candidate = fold["experiment"]["selected_models"][clock_report["primary_selection_prespecified"]]
        decision = fold["experiment"]["models"][candidate]
        clock_progress.append({
            "month": fold["name"], "candidate": candidate,
            "minimum_historical_recall": decision["minimum_validation_month_recall"],
            "historical_floor_met": decision["all_validation_months_meet_30_percent"],
            "development_recall": decision["development_evaluation"]["recall"],
        })
    display(pd.DataFrame(clock_progress))
    print("適用程度：", clock_report["suitability"])
    clock_groups = clock_report["selected_development_miss_diagnostics"]["aggregate"]["groups"]
    display(pd.DataFrame(clock_groups["state_route"]).head(10))
    display(pd.DataFrame(clock_groups["promise_days"]))
else:
    print("python scripts/improve_outcome_clock.py --n-jobs 4")


## 20. 可選：地理與組合特徵的同模型比較

本節只讀 `feature_engineering.json`，不啟動訓練、不新增圖表。四種模型以相同月份 deadline/wait0 訓練 cohort 與固定參數，比較既有 60 欄、地理＋組合特徵 83 欄，以及加出貨期限的 91 欄；LGB 的 60 欄版本重用，其餘新增 88 fit。距離與組合特徵一起加入，不能把差異全部歸因於距離。

主要選模 `primary_without_dispatch` 排除出貨期限；該欄缺少建立／修改時間，只在敏感度組比較。地理資料也沒有版本時間，假設是可於核准時查詢的靜態郵區參考；球面距離不是道路里程。新增特徵不使用訂單最終狀態或未來配送結果，歷史負載仍只用當時已知事件。

固定每日完整批次 10% 上限、三個歷史月最差召回優先選模；歷史 30% 門檻與開發月份表現分開呈現。本輪設計也受既有開發診斷啟發，所有 E 月仍是開發回測，原保留期間不重新評分。完整方法見 `docs/STAGE4_FEATURE_ENGINEERING.md`；未通過適用門檻前，維持研究模型與暫緩 Dashboard。


In [ ]:
ENGINEERING_PATH = PROJECT_ROOT / "reports" / "stage4" / "feature_engineering.json"
if ENGINEERING_PATH.is_file():
    engineering_report = json.loads(ENGINEERING_PATH.read_text(encoding="utf-8"))
    print("每日上限：", engineering_report["daily_cap"], "新增 fit：", engineering_report["new_fit_count"])
    print("主要選組：", engineering_report["primary_selection_prespecified"])
    display(pd.DataFrame(engineering_report["summary"])[[
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    paired_progress = []
    for item in engineering_report["paired_feature_comparisons"]:
        paired_progress.append({"模型": item["family"], **item["development_recall"],
            "83減60召回差": item["geo_minus_base_development_recall"],
            "91減83召回差（敏感度）": item["dispatch_minus_geo_development_recall"]})
    print("同模型、同訓練資料的開發回測；不依此表挑選部署模型：")
    display(pd.DataFrame(paired_progress))
    display(pd.DataFrame([engineering_report["suitability"]]))
else:
    print("尚無特徵工程報告；需要時執行 scripts/improve_feature_engineering.py。")


## 21. 可選：地理與負載交互消融

本節只讀 `feature_ablation.json`，不訓練、不新增圖表。保持四種模型的固定參數、deadline/wait0 訓練訂單與每日完整批次 10% 預算，比較既有 60 欄、只加地理 G 的 72 欄、只加負載交互 L 的 65 欄、G+L 的 77 欄，以及前輪完整 83 欄。

G 包含 9 個郵區位置／距離欄位與 3 個距離比例；L 包含 5 個待件壓力／逾期占比；剩餘 O 是 6 個成本、密度與日曆交互。60 欄基準已含原始待件計數、路線與 ZIP3，所以這不是移除所有地理或負載資訊。83 減 77 只是在 G+L 存在時加入 O 的差異，不是 O 單獨的效果；各差值也不是因果估計。

三個歷史 OOT 月決定選模，固定後才計算開發 E 月；這些月份與整體設計已受先前診斷影響，不是新獨立測試。地理資料缺少版本時間的限制保留，本輪不含出貨期限候選。未通過歷史逐月 30% 召回門檻前，維持研究 artifact、暫緩 Dashboard。完整方法見 `docs/STAGE4_FEATURE_ABLATION.md`。


In [ ]:
ABLATION_PATH = PROJECT_ROOT / "reports" / "stage4" / "feature_ablation.json"
if ABLATION_PATH.is_file():
    ablation_report = json.loads(ABLATION_PATH.read_text(encoding="utf-8"))
    print("固定每日上限：", ablation_report["daily_cap"], "新增 fit：", ablation_report["new_fit_count"])
    paired_rows = [{"模型": item["family"], **item["development_recall"]}
                   for item in ablation_report["paired_feature_comparisons"]]
    print("同模型、同訓練訂單的開發回測召回率；不依開發結果選模：")
    display(pd.DataFrame(paired_rows))
    ablation_summary = pd.DataFrame(ablation_report["summary"])
    display(ablation_summary.loc[ablation_summary.model_or_selection.isin(ablation_report["selection_groups"]), [
        "model_or_selection", "selected_candidates", "true_positives", "false_positives", "false_negatives",
        "precision", "recall", "minimum_validation_month_recall", "minimum_development_month_recall",
        "validation_folds_below_30_percent",
    ]])
    display(pd.DataFrame([ablation_report["suitability"]]))
else:
    print("尚無消融報告；需要時執行 scripts/ablate_delay_features.py。")


## 本階段留下的成果

- 可重用的訂單特徵、延遲標記、歷史與待件負載建置程式。
- 多月份 OOT、每月重訓、結果成熟篩選與每日 10% 提醒預算。
- 同模型特徵工程比較、成對結果與完整失敗紀錄。
- 預設模型與研究 artifact 分開保留，附完整日批次推論介面。

目前持續改善固定預算召回；先確認歷史月份的適用門檻與後續新期間表現，再進行 Dashboard。開發回測不能當成未知資料成績。
